### Baseline Model Development

Goal:
Build the first baseline model for loan risk prediction using the cleaned dataset.

#### Baseline Model Development

Goal:
Predict whether a borrower will default on a loan.

Target:
is_default
0 = Fully Paid
1 = Charged Off / Default

Current Dataset:
X.shape = (87889, 89)
y.shape = (87889,)

Planned Baseline:
Logistic Regression

Evaluation Strategy:
- Stratified Train/Test Split
- Accuracy not used as primary metric
- Focus on classification metrics suitable for imbalanced data

## Next Steps

- Perform stratified 80/20 train-test split
- Train Logistic Regression baseline
- Evaluate using ROC-AUC, Precision, Recall, F1-score
- Compare with future tree-based models

In [ ]:
import pandas as pd

In [ ]:
X = pd.read_csv("../data/processed/X.csv", index_col=0)

In [ ]:
X.shape

In [ ]:
y =pd.read_csv("../data/processed/y.csv", index_col=0).squeeze()

In [ ]:
y.shape

In [ ]:
X.index.equals(y.index)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
X.dtypes.value_counts()

In [ ]:
X.select_dtypes(include="bool").columns.tolist()


In [ ]:
X.select_dtypes(include="int64").columns.to_list()

In [ ]:
y.value_counts()

In [ ]:
X["is_default"].value_counts()

In [ ]:
X[['term', 'sub_grade', 'verification_status',
   'initial_list_status', 'application_type',
   'debt_settlement_flag']].nunique()

In [ ]:
X[['term', 'sub_grade', 'verification_status',
   'initial_list_status', 'application_type',
   'debt_settlement_flag']].head()

In [ ]:
X.select_dtypes(include="float64").columns.tolist()

In [ ]:
X.select_dtypes(include="float64").nunique().sort_values().to_frame("n_unique")

In [ ]:
X.select_dtypes(include="float64").nunique().sort_values().head(15)

In [ ]:
X.select_dtypes(include="float64").nunique().sort_values().head(25)

In [ ]:
X_train.dtypes.value_counts()

In [ ]:
excluded_cols = [
    "id",
    "is_default",
    "debt_settlement_flag",
    "policy_code",
    "out_prncp",
    "out_prncp_inv",
]

In [ ]:
[col for col in [
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
] if col in X.columns]

In [ ]:
X_train = X_train.drop(columns=excluded_cols)
X_test = X_test.drop(columns=excluded_cols)

In [ ]:
[col for col in excluded_cols if col in X_train.columns or col in X_test.columns]

In [ ]:
numerical_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
numerical_cols

In [ ]:
binary_cols = X_train.select_dtypes(include=["bool"]).columns.to_list()
binary_cols

In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.to_list()
categorical_cols

In [ ]:
categorical_cols = ["verification_status"]

In [ ]:
"verification_status" in numerical_cols

In [ ]:
numerical_cols.remove("verification_status")

In [ ]:
"verification_status" in numerical_cols

In [ ]:
low_cardinality = (
    X_train.nunique()
    .sort_values()
    .loc[lambda s: (s >= 3) & (s <= 12)]
)

low_cardinality

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(), categorical_cols),
        ("bin", "passthrough", binary_cols)
    ],
    remainder="drop"
)

In [ ]:
preprocessor.fit(X_train)

In [ ]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
X_train_transformed.shape, X_test_transformed.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
model.fit(X_train_transformed, y_train)

In [ ]:
y_test_proba = model.predict_proba(X_test_transformed)

In [ ]:
y_test_proba.shape

In [ ]:
y_test_pred = model.predict(X_test_transformed)

In [ ]:
y_test_pred.shape

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)
cm

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision =  precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

precision, recall

In [ ]:
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
from sklearn.metrics import f1_score

f1 = f1_score(y_test, y_test_pred)
f1

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_test_pred)

accuracy

In [ ]:
y_test_default_proba = y_test_proba[:, 1]


In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_test, y_test_default_proba)
roc_auc

In [ ]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

In [ ]:
y_test_pred_30 = (y_test_default_proba >= 0.30).astype(int)

In [ ]:
y_test_pred_30.shape

In [ ]:
y_test_pred.sum()


In [ ]:
y_test_pred_30.sum()

In [ ]:
cm_30 = confusion_matrix(y_test, y_test_pred_30)
cm_30

In [ ]:
threshold_results = []

for threshold in thresholds:
    y_test_pred_threshold = (y_test_default_proba >= threshold).astype(int)
    cm_threshold = confusion_matrix(y_test, y_test_pred_threshold)
    tn = cm_threshold[0][0]
    fp = cm_threshold[0][1]
    fn = cm_threshold[1][0]
    tp = cm_threshold[1][1]

    precision_threshold = tp / (tp + fp)
    recall_threshold = tp / (tp + fn)

    f1_score_threshold = 2 * ((precision_threshold * recall_threshold) / (precision_threshold + recall_threshold))

    results = {"threshold": threshold,
               "precision": precision_threshold,
                "recall" : recall_threshold,
                "f1 score" : f1_score_threshold,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn}

    threshold_results.append(results)


threshold_df = pd.DataFrame(threshold_results)
threshold_df

In [ ]:
false_negative_mask = (y_test == 1) & (y_test_pred == 0)
false_negative_mask.sum()

In [ ]:
X_false_negatives = X_test[false_negative_mask]
X_false_negatives.shape

In [ ]:
true_positive_mask = (y_test == 1) & (y_test_pred == 1)
true_positive_mask.sum()

In [ ]:
X_true_positives = X_test[true_positive_mask]
X_true_positives.shape

In [ ]:
X_test.head()

In [ ]:
features = ["annual_inc", "loan_amnt", "int_rate", "dti"]

comparison = pd.DataFrame({
    "True Positives": X_true_positives[features].median(),
    "False Negatives": X_false_negatives[features].median()
})

comparison

### Error Analysis: False Negatives vs True Positives

We compared the median values of annual income, loan amount, interest rate, and debt-to-income ratio (DTI) between correctly identified defaulters (True Positives) and missed defaulters (False Negatives).

**Observations:**
- Annual income: The median values are very close (61,000 vs 60,000).
- Loan amount: There is a small difference (15,000 vs 12,000).
- Interest rate: The median values are relatively close (13.99 vs 12.59).
- DTI: Both groups have almost identical median values (20.78 vs 20.945).

**Conclusion:**

The median values of these four features do not reveal a strong distinction between True Positives and False Negatives.

This suggests that comparing individual feature medians alone may not be sufficient to understand why the model misses certain defaulters. Further investigation of feature distributions and interactions may be required.

In [ ]:
false_positive_mask = (y_test == 0) & (y_test_pred == 1)
false_positive_mask.sum()

In [ ]:
X_false_positives = X_test[false_positive_mask]
X_false_positives.shape

In [ ]:
true_negative_mask = (y_test == 0) & (y_test_pred == 0)
true_negative_mask.sum()

In [ ]:
X_true_negatives = X_test[true_negative_mask]
X_true_negatives.shape

In [ ]:
features = ["annual_inc", "loan_amnt", "int_rate", "dti"]

comparison = pd.DataFrame({
    "False Positives": X_false_positives[features].median(),
    "True Negatives": X_true_negatives[features].median()
})

comparison

### Error Analysis: False Positives vs True Negatives

We compared the median values of annual income, loan amount, interest rate, and debt-to-income ratio (DTI) between incorrectly flagged borrowers (False Positives) and correctly identified non-defaulters (True Negatives).

**Observations:**
- Annual income: False Positives have a lower median annual income (53,500 vs 67,000).
- Loan amount: False Positives have a lower median loan amount (10,000 vs 12,000).
- Interest rate: False Positives have a higher median interest rate (13.18 vs 10.99).
- DTI: False Positives have a higher median DTI (19.51 vs 17.72).

**Conclusion:**

False Positives show some differences in borrower characteristics compared with True Negatives. In particular, they have lower median income and higher median interest rates and DTI.

These characteristics may be associated with the model incorrectly classifying some non-defaulters as defaulters. However, median comparisons alone cannot establish the reasons behind individual misclassifications.